# 📄 Document Loaders & Text Splitters — LangChain Bangla

RAG (Retrieval-Augmented Generation) সিস্টেম তৈরির প্রথম ধাপ হলো ডকুমেন্ট লোড করা এবং সেগুলো ছোট ছোট অংশে ভাগ করা।

LLM সরাসরি পুরো বই বা বড় ফাইল পড়তে পারে না — কারণ তার **context window** সীমিত।  
তাই আমরা ডকুমেন্টকে ছোট **chunk**-এ ভাগ করি এবং শুধু প্রাসঙ্গিক অংশটুকু LLM-কে দিই।

---

## এই নোটবুকে যা শিখব:

| বিষয় | বিবরণ |
|-------|-------|
| `TextLoader` | `.txt` ফাইল লোড করা |
| `PyPDFLoader` | PDF ফাইল লোড করা |
| `WebBaseLoader` | ওয়েবপেজ থেকে content লোড করা |
| `RecursiveCharacterTextSplitter` | স্মার্টভাবে একাধিক separator দিয়ে ভাগ করা |
| `CharacterTextSplitter` | একটি নির্দিষ্ট separator দিয়ে ভাগ করা |

---

## পুরো প্রক্রিয়া এক নজরে:

```
ফাইল / ওয়েবপেজ / PDF
        ↓  Document Loader
  Document Object
        ↓  Text Splitter
  ছোট Chunks
        ↓  (পরের নোটবুকে)
  Vector Store → Retrieval → LLM
```


## RAG করার আগে যা করতে হবে

RAG মানে হলো — LLM-কে উত্তর দেওয়ার আগে প্রাসঙ্গিক তথ্য খুঁজে বের করে দেওয়া।  
এই কাজটি করতে হলে প্রথমে ৩টি ধাপ পার করতে হয়:

### ধাপ ১: ডকুমেন্ট লোড করো
বিভিন্ন উৎস থেকে ডকুমেন্ট পড়তে হবে — যেমন PDF, TXT ফাইল, বা ওয়েবপেজ।

### ধাপ ২: ডকুমেন্ট ভাগ করো
বড় ডকুমেন্টকে ছোট ছোট অংশে (chunk) ভাগ করতে হবে।

### ধাপ ৩: Chunk সংরক্ষণ করো
পরে search করার জন্য এই chunk গুলো Vector Store-এ রাখতে হবে।

## কেন ভাগ করতে হয়?

LLM-এর context window সীমিত — পুরো বই একসাথে দেওয়া সম্ভব না।  
তাই ডকুমেন্টকে ছোট chunk-এ ভাগ করে শুধু প্রাসঙ্গিক অংশটুকু খুঁজে নিতে হয়।

```
Load → Split → Store → Retriever
```


## 📂 Document Loader কী?

LangChain সরাসরি raw ফাইল (PDF, TXT, CSV) নিয়ে কাজ করতে পারে না।  
**Document Loader** সেই ফাইলগুলোকে LangChain-এর নিজস্ব `Document` অবজেক্টে রূপান্তর করে।

```
Raw File (PDF/TXT/Web)  →  Document Loader  →  Document Object
                                                  ├── page_content  (মূল টেক্সট)
                                                  └── metadata      (ফাইলের তথ্য)
```

## কোন Loader কখন ব্যবহার করবেন:

| Loader | কী লোড করে | Import |
|--------|-----------|--------|
| `PyPDFLoader` | PDF ফাইল | `from langchain_community.document_loaders import PyPDFLoader` |
| `TextLoader` | TXT / plain text | `from langchain_community.document_loaders import TextLoader` |
| `CSVLoader` | CSV ফাইল | `from langchain_community.document_loaders import CSVLoader` |
| `UnstructuredWordDocumentLoader` | DOCX / Word ফাইল | `from langchain_community.document_loaders import UnstructuredWordDocumentLoader` |
| `UnstructuredExcelLoader` | Excel (.xlsx) | `from langchain_community.document_loaders import UnstructuredExcelLoader` |
| `JSONLoader` | JSON ফাইল | `from langchain_community.document_loaders import JSONLoader` |
| `DirectoryLoader` | একটি ফোল্ডারের সব ফাইল | `from langchain_community.document_loaders import DirectoryLoader` |
| `WebBaseLoader` | ওয়েবপেজ / URL | `from langchain_community.document_loaders import WebBaseLoader` |
| `WikipediaLoader` | Wikipedia আর্টিকেল | `from langchain_community.document_loaders import WikipediaLoader` |
| `YoutubeLoader` | YouTube transcript | `from langchain_community.document_loaders import YoutubeLoader` |


### ১. TextLoader — সাধারণ টেক্সট ফাইল লোড করা

`TextLoader` দিয়ে যেকোনো `.txt` ফাইল লোড করা যায়।  
লোড করার পর `docs` হলো একটি list — প্রতিটি element একটি `Document` অবজেক্ট।  
- `docs[0].page_content` → ফাইলের মূল টেক্সট
- `docs[0].metadata` → ফাইলের path ও অন্যান্য তথ্য


In [1]:
from langchain_community.document_loaders import TextLoader

text_loader= TextLoader("../Data/Examples/example.txt", encoding="utf-8")

docs= text_loader.load()


print(f" Loaded {len(docs)} documents")
print(f" Document Type {type(docs[0])}")
print(f" Content Preview:  {docs[0].page_content[:100]}...")
print(f"Metadata: {docs[0].metadata}")

 Loaded 1 documents
 Document Type <class 'langchain_core.documents.base.Document'>
 Content Preview:  
What are Document Loaders?
Document Loader is one of the components of the LangChain framework. It ...
Metadata: {'source': '../Data/Examples/example.txt'}


### ২. PyPDFLoader — PDF ফাইল লোড করা

`PyPDFLoader` দিয়ে PDF ফাইল লোড করা হয়।  
**বিশেষত্ব:** PDF-এর প্রতিটি পেজ আলাদা `Document` হিসেবে আসে।  
তাই ১০ পেজের PDF লোড করলে `docs`-এ ১০টি element থাকবে।

> 💡 **Install:** `pip install pypdf`


In [2]:
from langchain_community.document_loaders import PyPDFLoader

pdf_loader= PyPDFLoader("../Data/Examples/example.pdf")
docs= pdf_loader.load()


print(f" Loaded {len(docs)} documents")
print(f" Document Type {type(docs[0])}")
print(f" Content Preview:  {docs[0].page_content[:100]}...")
print(f"Metadata: {docs[0].metadata}")

 Loaded 5 documents
 Document Type <class 'langchain_core.documents.base.Document'>
 Content Preview:  What are Document Loaders?
Document Loader is one of the components of the LangChain framework. It i...
Metadata: {'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'source': '../Data/Examples/example.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}


### ৩. WebBaseLoader — ওয়েবপেজ থেকে content লোড করা

`WebBaseLoader` দিয়ে যেকোনো URL থেকে টেক্সট content লোড করা যায়।  
এখানে `bs4.SoupStrainer` ব্যবহার করে শুধু Wikipedia-র মূল content অংশ (`mw-content-text`) নেওয়া হচ্ছে — বাকি header, footer, sidebar বাদ দেওয়া হচ্ছে।

> 💡 **Install:** `pip install beautifulsoup4 requests`


In [3]:
import os
import bs4
from langchain_community.document_loaders import WebBaseLoader

os.environ["USER_AGENT"] = "my-langchain-app"

url = "https://en.wikipedia.org/wiki/LangChain"

web_loader = WebBaseLoader(
    web_paths=[url],
    bs_kwargs={
        "parse_only":bs4.SoupStrainer(id="mw-content-text")
    }
    )


docs = web_loader.load()


print(f" Loaded {len(docs)} documents")
print(f" Document Type {type(docs[0])}")
print(f" Content Preview:  {docs[0].page_content[:100]}...")
print(f"Metadata: {docs[0].metadata}")


USER_AGENT environment variable not set, consider setting it to identify your requests.


 Loaded 1 documents
 Document Type <class 'langchain_core.documents.base.Document'>
 Content Preview:  
Language model application development framework
LangChainDeveloperHarrison ChaseInitial releaseOct...
Metadata: {'source': 'https://en.wikipedia.org/wiki/LangChain'}


> ✅ **মূল কথা:** Document Loader raw ফাইলকে `Document` অবজেক্টে রূপান্তর করে।  
> প্রতিটি Document-এ দুটি জিনিস থাকে — `page_content` (মূল টেক্সট) এবং `metadata` (উৎসের তথ্য)।


---

# ✂️ Text Splitters — ডকুমেন্ট ভাগ করা

Document লোড করার পর পরের কাজ হলো সেটিকে ছোট **chunk**-এ ভাগ করা।  
এই chunk গুলো পরে Vector Store-এ রাখা হয় এবং similarity search করা হয়।


## মূল ধারণা

বড় ডকুমেন্টকে ছোট অংশে ভাগ করা হয় — এই ছোট অংশগুলোকে **chunk** বলে।

```
বড় Document  →  [chunk 1] [chunk 2] [chunk 3] ...
```

## Chunk Overlap — কেন গুরুত্বপূর্ণ?

দুটি chunk-এর মাঝে কিছু অংশ **ইচ্ছাকৃতভাবে repeat** করা হয়।  
কারণ chunk-এর শুরু বা শেষে গুরুত্বপূর্ণ তথ্য কেটে যেতে পারে — overlap সেটা রোধ করে।

```
raw_text :  "AAAA BBBB CCCC DDDD EEEE FFFF GGGG"
chunk 1:   [ AAAA BBBB CCCC ]
chunk 2:              [ CCCC DDDD EEEE ]   ← CCCC repeat হয়েছে (overlap)
chunk 3:                         [ EEEE FFFF GGGG ]  ← EEEE repeat হয়েছে
```

> 💡 Overlap না থাকলে chunk-এর boundary-তে context হারিয়ে যেতে পারে।


# CharacterTextSplitter বনাম RecursiveCharacterTextSplitter

---

## CharacterTextSplitter

এটি শুধুমাত্র **একটি separator** দিয়ে text ভাগ করে।

```python
CharacterTextSplitter(
    chunk_size=200,
    separator="\n"
)
```

### কীভাবে কাজ করে:
- শুধু newline (`\n`) দিয়ে text ভাগ করে
- তারপর অংশগুলো জোড়া লাগায় যতক্ষণ chunk size সীমায় পৌঁছায়

### সীমাবদ্ধতা:
`chunk_size` একটি নিশ্চিত গ্যারান্টি না।  
যদি একটি line-ই ৫০০ character-এর বেশি হয় এবং ভেতরে কোনো `\n` না থাকে — তাহলে সেই পুরো line একটি chunk হয়ে যাবে, যদিও `chunk_size=200` দেওয়া আছে।

```
Warning: Created a chunk of size 1520, which is longer than the specified 200
```

---

## RecursiveCharacterTextSplitter

এটি **একাধিক separator** ধাপে ধাপে ব্যবহার করে — তাই অনেক স্মার্ট।

```python
RecursiveCharacterTextSplitter(
    chunk_size=200,
    separators=["\n\n", "\n", ".", " ", ""]
)
```

### কীভাবে কাজ করে (ধাপে ধাপে):
1. প্রথমে paragraph দিয়ে ভাগ করার চেষ্টা করে (`\n\n`)
2. তারপর line দিয়ে (`\n`)
3. তারপর বাক্য দিয়ে (`.`)
4. তারপর শব্দ দিয়ে (` `)
5. সবশেষে character দিয়ে (`""`)

একটি পদ্ধতিতে chunk size limit না মানলে পরের ছোট separator দিয়ে আবার চেষ্টা করে।

---

## কেন Recursive Splitter ভালো?

✅ chunk size অনেক ভালোভাবে মেনে চলে  
✅ oversized chunk কমে যায়  
✅ context স্বাভাবিকভাবে ধরে রাখে  
✅ RAG ও embedding-এর জন্য আদর্শ  

---

## পার্থক্য এক নজরে:

| বৈশিষ্ট্য | CharacterTextSplitter | RecursiveCharacterTextSplitter |
|-----------|----------------------|--------------------------------|
| Separator সংখ্যা | একটি | একাধিক |
| স্মার্ট fallback | ❌ নেই | ✅ আছে |
| chunk size নিশ্চয়তা | গ্যারান্টি নেই | অনেক ভালো নিয়ন্ত্রণ |
| oversized chunk সমস্যা | প্রায়ই হয় | খুব কমই হয় |
| কখন ব্যবহার করবেন | সহজ text-এ | RAG / LLM app-এ |

---

> ⚠️ **মনে রাখো:** `chunk_size = 200` মানে এই না যে প্রতিটি chunk ঠিক ২০০ character হবে।  
> এর মানে হলো — "যতটা সম্ভব ২০০-এর কাছাকাছি রাখার চেষ্টা করো।"  
> শুধু `RecursiveCharacterTextSplitter` এটা কার্যকরভাবে enforce করতে পারে।


### RecursiveCharacterTextSplitter ব্যবহার করা

WebBaseLoader দিয়ে লোড করা Wikipedia page-টি এখন chunk-এ ভাগ করা হচ্ছে।  
- `chunk_size=200` → প্রতিটি chunk সর্বোচ্চ ২০০ character
- `chunk_overlap=30` → দুটি chunk-এর মাঝে ৩০ character repeat হবে


In [4]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter


splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap= 30,
    separators= [""] # "\n\n", "\n", ".", " ", 
)

chunks= splitter.split_documents(docs)
print(f"Original docs: {len(docs)}")
print(f"After Splitting: {len(chunks)} chunks")


Original docs: 1
After Splitting: 94 chunks


### প্রথম ৫টি Chunk দেখা

এখানে loop করে প্রথম ৫টি chunk print করা হচ্ছে —  
দেখতে পাবেন প্রতিটি chunk কত ছোট এবং কীভাবে ভাগ হয়েছে।


In [5]:
for i , chunk in enumerate(chunks):
    print(f"Chunks--> {i+1} ---> \n {chunk.page_content}")
    if i==4:
        break

Chunks--> 1 ---> 
 Language model application development framework
LangChainDeveloperHarrison ChaseInitial releaseOctober 2022Stable release0.1.16[1]
   / 11 April 2024; 2 years ago (11 April 2024)
Written inPython an
Chunks--> 2 ---> 
 pril 2024)
Written inPython and JavaScriptTypeSoftware framework for large language model application developmentLicenseMIT LicenseWebsiteLangChain.comRepositorygithub.com/langchain-ai/langchain

Free
Chunks--> 3 ---> 
 m/langchain-ai/langchain

Free and open-source software portal
LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language
Chunks--> 4 ---> 
 o applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and
Chunks--> 5 ---> 
 summarization, chatbots, and code analysis.[2]


History[edit]
LangChain was launched in October 2022 as

### CharacterTextSplitter ব্যবহার করা

এবার তুলনার জন্য `CharacterTextSplitter` দিয়ে একই document ভাগ করা হচ্ছে।  
- শুধু `\n` (newline) separator ব্যবহার করা হচ্ছে
- `chunk_overlap=0` → কোনো overlap নেই

Recursive splitter-এর সাথে chunk সংখ্যা তুলনা করে দেখো — পার্থক্য বুঝতে পারবে।


In [12]:

chara_splitter = CharacterTextSplitter(
    chunk_size=200,
    chunk_overlap= 0,
    separator= "\n" # "\n\n", "\n", ".", " ", 
)

chunks= chara_splitter.split_documents(docs)
print(f"Original docs: {len(docs)}")
print(f"After Splitting: {len(chunks)} chunks")

Created a chunk of size 324, which is longer than the specified 200
Created a chunk of size 394, which is longer than the specified 200
Created a chunk of size 355, which is longer than the specified 200
Created a chunk of size 612, which is longer than the specified 200
Created a chunk of size 1520, which is longer than the specified 200
Created a chunk of size 217, which is longer than the specified 200
Created a chunk of size 202, which is longer than the specified 200
Created a chunk of size 202, which is longer than the specified 200


Original docs: 1
After Splitting: 79 chunks


### প্রথম ৫টি Chunk-এর বিস্তারিত দেখা

`chunks[:5]` দিয়ে প্রথম ৫টি `Document` object দেখা যাচ্ছে —  
প্রতিটিতে `page_content` এবং `metadata` আছে।


In [13]:
chunks[:5]

[Document(metadata={'source': 'https://en.wikipedia.org/wiki/LangChain'}, page_content='Language model application development framework\nLangChainDeveloperHarrison ChaseInitial releaseOctober 2022Stable release0.1.16[1]\n   / 11 April 2024; 2 years ago\xa0(11 April 2024)'),
 Document(metadata={'source': 'https://en.wikipedia.org/wiki/LangChain'}, page_content='Written inPython and JavaScriptTypeSoftware framework for large language model application developmentLicenseMIT LicenseWebsiteLangChain.comRepositorygithub.com/langchain-ai/langchain'),
 Document(metadata={'source': 'https://en.wikipedia.org/wiki/LangChain'}, page_content='Free and open-source software portal'),
 Document(metadata={'source': 'https://en.wikipedia.org/wiki/LangChain'}, page_content="LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models

> ✅ **মূল কথা:** RAG-এ সবসময় `RecursiveCharacterTextSplitter` ব্যবহার করা ভালো।  
> এটি স্মার্টভাবে একাধিক separator দিয়ে text ভাগ করে এবং chunk size অনেক ভালো নিয়ন্ত্রণে রাখে।
